# Phase 2: Baseline Text Classifier
**Project:** AI Mental Health Support Chatbot  
**Goal:** Train and evaluate classical NLP baselines (TF-IDF + Logistic Regression and LinearSVC) to establish benchmarks for downstream DistilBERT fine-tuning.

In [ ]:
import sys
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.append("..")
from src.models.baseline import (
    create_stratified_splits,
    get_canonical_labels,
    save_reproducibility_metadata,
    fit_tfidf_vectorizer,
    train_logistic_regression,
    train_linear_svc,
    evaluate_model,
    plot_and_save_confusion_matrix,
    save_model_artifact,
    load_model_artifact
)

## 1. Load Cleaned Dataset & Verify Data
We load `data/processed/cleaned_mental_health_data.csv` created in Phase 1.

In [ ]:
df_clean = pd.read_csv("../data/processed/cleaned_mental_health_data.csv")
print(f"Dataset shape: {df_clean.shape}")
print("Null count:\n", df_clean.isnull().sum())
print("\nClass Distribution:")
print(df_clean['status'].value_counts())

## 2. Create Stratified Data Splits (80/10/10)
We split the data into 80% Train, 10% Validation, and 10% Test using stratified sampling with `random_state=42`.

In [ ]:
train_df, val_df, test_df = create_stratified_splits(
    df_path="../data/processed/cleaned_mental_health_data.csv",
    output_dir="../data/processed/splits",
    random_state=42
)
print(f"Train size: {len(train_df)} ({len(train_df)/len(df_clean)*100:.2f}%)")
print(f"Val size:   {len(val_df)} ({len(val_df)/len(df_clean)*100:.2f}%)")
print(f"Test size:  {len(test_df)} ({len(test_df)/len(df_clean)*100:.2f}%)")

## 3. Verify Data Leakage & Class Distributions across Splits

In [ ]:
# Verify zero statement overlap
train_set = set(train_df['statement'])
val_set = set(val_df['statement'])
test_set = set(test_df['statement'])
assert len(train_set.intersection(val_set)) == 0
assert len(train_set.intersection(test_set)) == 0
assert len(val_set.intersection(test_set)) == 0
print("Zero statement overlap confirmed across train, val, and test splits!")

split_dist = pd.DataFrame({
    "Train": train_df['status'].value_counts(),
    "Validation": val_df['status'].value_counts(),
    "Test": test_df['status'].value_counts()
})
print("\nClass distributions across splits:")
print(split_dist)

## 4. Fit TF-IDF Vectorizer (Training Set ONLY)
We fit `TfidfVectorizer` with `ngram_range=(1,2)`, `min_df=2`, `max_df=0.95`, `sublinear_tf=True` exclusively on `train_df['statement']`.

In [ ]:
vectorizer = fit_tfidf_vectorizer(train_df['statement'])
X_train = vectorizer.transform(train_df['statement'])
X_val = vectorizer.transform(val_df['statement'])
X_test = vectorizer.transform(test_df['statement'])

print(f"TF-IDF Vocabulary Size: {len(vectorizer.vocabulary_)} features")
save_model_artifact(vectorizer, "../models/baseline/tfidf_vectorizer.joblib")

## 5. Train & Evaluate Baseline 1: Logistic Regression

In [ ]:
labels = get_canonical_labels()
lr_model = train_logistic_regression(X_train, train_df['status'])
save_model_artifact(lr_model, "../models/baseline/logistic_regression.joblib")

lr_val = evaluate_model(lr_model, X_val, val_df['status'], labels)
print(f"Logistic Regression Validation Accuracy: {lr_val['accuracy']:.4f}")
print(f"Logistic Regression Validation Macro F1: {lr_val['macro_f1']:.4f}")
print(f"Logistic Regression Validation Weighted F1: {lr_val['weighted_f1']:.4f}")

## 6. Train & Evaluate Baseline 2: LinearSVC

In [ ]:
svc_model = train_linear_svc(X_train, train_df['status'])
save_model_artifact(svc_model, "../models/baseline/linear_svc.joblib")

svc_val = evaluate_model(svc_model, X_val, val_df['status'], labels)
print(f"LinearSVC Validation Accuracy: {svc_val['accuracy']:.4f}")
print(f"LinearSVC Validation Macro F1: {svc_val['macro_f1']:.4f}")
print(f"LinearSVC Validation Weighted F1: {svc_val['weighted_f1']:.4f}")

## 7. Model Comparison & Baseline Selection
We compare Logistic Regression and LinearSVC based on **Validation Macro F1**.

In [ ]:
comp_df = pd.DataFrame({
    "Metric": ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1"],
    "Logistic Regression": [lr_val['accuracy'], lr_val['macro_precision'], lr_val['macro_recall'], lr_val['macro_f1'], lr_val['weighted_f1']],
    "LinearSVC": [svc_val['accuracy'], svc_val['macro_precision'], svc_val['macro_recall'], svc_val['macro_f1'], svc_val['weighted_f1']]
}).set_index("Metric")
print(comp_df.round(4))

if lr_val['macro_f1'] >= svc_val['macro_f1']:
    winning_name = "Logistic Regression"
    winning_model = lr_model
else:
    winning_name = "LinearSVC"
    winning_model = svc_model

print(f"\nSelected Winning Baseline: {winning_name}")

## 8. Evaluate Selected Model ONCE on Test Set

In [ ]:
test_eval = evaluate_model(winning_model, X_test, test_df['status'], labels)
print(f"Final Test Accuracy ({winning_name}):    {test_eval['accuracy']:.4f}")
print(f"Final Test Macro F1 ({winning_name}):    {test_eval['macro_f1']:.4f}")
print(f"Final Test Weighted F1 ({winning_name}): {test_eval['weighted_f1']:.4f}")

print("\nTest Set Per-Class Metrics:")
test_per_class = pd.DataFrame(test_eval['per_class']).T
print(test_per_class.round(4))

## 9. Plot Confusion Matrices

In [ ]:
plot_and_save_confusion_matrix(
    svc_val['confusion_matrix'], labels,
    "Validation Confusion Matrix - LinearSVC",
    "../models/baseline/cm_val_linear_svc.png"
)
plot_and_save_confusion_matrix(
    test_eval['confusion_matrix'], labels,
    f"Test Confusion Matrix - {winning_name}",
    "../models/baseline/cm_test_selected_baseline.png"
)
print("Confusion matrix plots generated and saved!")

## 10. Save Reproducibility Metadata & Summary
Save experiment configuration to `models/baseline/metadata.json`.

In [ ]:
metadata = save_reproducibility_metadata("../models/baseline/metadata.json", len(df_clean))
print("Metadata saved successfully:")
print(json.dumps(metadata, indent=2))